In [1]:
%pip install numpy pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 44.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 55.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 44.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 45.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# 1. Load the dataset (Assuming the CSV is in your current directory)
df = pd.read_csv('Crime_Data_from_2020_to_Present.csv')

# 2. Filter for specific Crime Codes (442 and 343)
# We use .isin() for a clean, readable filter
assault_codes = [230, 231, 235, 236, 624, 625, 626, 627]
df_filtered = df[df['Crm Cd'].isin(assault_codes)].copy()

# 3. Remove columns with significant nulls
null_pct = (df_filtered.isnull().sum() / len(df_filtered) * 100).round(2)
threshold = 80
columns_to_drop = null_pct[null_pct > threshold].index.tolist()
df_filtered = df_filtered.drop(columns=columns_to_drop)
print(f"\nDropped columns: {columns_to_drop}")

# 4. Clean the Date Columns (Removing the '12:00:00 AM' portion)
# Using .dt.date effectively 'picks' only the YYYY-MM-DD
df_filtered['Date Rptd'] = pd.to_datetime(df_filtered['Date Rptd']).dt.date
df_filtered['DATE OCC'] = pd.to_datetime(df_filtered['DATE OCC']).dt.date

# 5. Convert 'TIME OCC' to a standard Time format
# In this dataset, 1:30 PM is usually '1330'. We pad with zeros and format it.
df_filtered['TIME OCC'] = pd.to_datetime(df_filtered['TIME OCC'].astype(str).str.zfill(4), 
                                        format='%H%M').dt.time

# 6. Handle missing values in spatial coordinates (as mentioned in your risks)
# Removing rows where Lat/Lon are 0.0 or NaN
df_filtered = df_filtered[(df_filtered['LAT'] != 0) & (df_filtered['LON'] != 0)]
df_filtered.dropna(subset=['LAT', 'LON'], inplace=True)

# 7. Add Descriptive Labels (Optional but helpful for EDA)
code_map = {
    230: "ADW / Aggravated Assault",
    231: "ADW on Police Officer",
    235: "Child Abuse (Aggravated)",
    236: "Spousal Abuse (Aggravated)",
    624: "Battery - Simple Assault",
    625: "Other Assault",
    626: "Spousal Abuse (Simple)",
    627: "Child Abuse (Simple)"
}
df_filtered['Assault Category'] = df_filtered['Crm Cd'].map(code_map)

# 8. Encoding 'Vict Sex' (One-Hot Encoding)
# Using sparse_output=False makes it much easier to convert to a DataFrame
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
vict_sex_encoded = encoder.fit_transform(df_filtered[['Vict Sex']].fillna('Unknown'))

# Create DataFrame from encoded variables
df_one_hot = pd.DataFrame(
    vict_sex_encoded, 
    columns=encoder.get_feature_names_out(['Vict Sex']),
    index=df_filtered.index # Important: keeps the rows aligned!
)

# Join the dummy variables back to our main dataset
data_final = pd.concat([df_filtered, df_one_hot], axis=1)

# 9. Quick overview of the filtered data
print(f"Total Assault Incidents: {len(df_filtered)}")
print(df_filtered['Assault Category'].value_counts())

# Save to a new CSV for your team
df_filtered.to_csv('LA_Assault_Data_2020_Present.csv', index=False)


Dropped columns: ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4']


/tmp/ipykernel_2758/3108333348.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_filtered['Date Rptd'] = pd.to_datetime(df_filtered['Date Rptd']).dt.date
/tmp/ipykernel_2758/3108333348.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_filtered['DATE OCC'] = pd.to_datetime(df_filtered['DATE OCC']).dt.date


Total Assault Incidents: 196594
Assault Category
Battery - Simple Assault      74483
ADW / Aggravated Assault      53504
Spousal Abuse (Simple)        46489
Spousal Abuse (Aggravated)    12652
Other Assault                  4219
Child Abuse (Simple)           3555
ADW on Police Officer          1078
Child Abuse (Aggravated)        614
Name: count, dtype: int64
